In [1]:
using MarineHydro
using PyCall
using StaticArrays
using DifferentiationInterface
import DifferentiationInterface as DI
using Enzyme
using Printf



SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


In [2]:
mesh_from_cpt = false
static_arrays_mesh = false

false

In [3]:
if mesh_from_cpt
    cpt = pyimport("capytaine")
    function mesh_r(r)
        Mesh(cpt.mesh_sphere(
            name="sphere", radius=r, center=(0.0, 0.0, 0.0),
            resolution=(6, 6)).immersed_part())
    end
    fd_mesh_rules!(mesh_r)
else  
    function mesh_r(r)
        d1=0.16
        d2=0.37
        # r1=0.88
        r2=0.35
        wavebot_mesh(r,r2,d1,d2,(3, 5, 3),10)
    end
end

if static_arrays_mesh
    mesh_r_(r) = StaticArraysMesh(mesh_r(r))
else
    mesh_r_(r) = mesh_r(r)
end

body(r) = FloatingBody(mesh_r_(r), [:Heave], "floatingbody")



const formulation = DirectBEM()                # default: direct + Wu
const r0 = 1.0
const ω0 = 1.03
const β0 = 0.0
const ωs = [0.8, 1.03, 1.4]
const fwd = AutoForwardDiff()
const rev = AutoEnzyme(; mode=Enzyme.set_runtime_activity(Enzyme.Reverse))


radiation_solve(r, ω, form=formulation) = solve(RadiationProblem(body(r), ω), form)

sol = radiation_solve(r0, ω0)
@printf("A₃₃ = %.6f\nB₃₃ = %.6f\n", added_mass(sol).Heave, radiation_damping(sol).Heave)

omega: 1.03, wavenumber: 0.10814475025484199
mesh normals: Mesh{Float64, Matrix{Float64}, Matrix{Int64}, Matrix{Float64}, Matrix{Float64}, Vector{Float64}, Vector{Float64}}([1.0 0.0 -0.0; 0.8090169943749475 0.5877852522924731 -0.0; 0.30901699437494745 0.9510565162951535 -0.0; -0.30901699437494734 0.9510565162951536 -0.0; -0.8090169943749473 0.5877852522924732 -0.0; -1.0 1.2246467991473532e-16 -0.0; -0.8090169943749475 -0.587785252292473 -0.0; -0.30901699437494756 -0.9510565162951535 -0.0; 0.30901699437494723 -0.9510565162951536 -0.0; 0.8090169943749473 -0.5877852522924734 -0.0; 1.0 0.0 -0.08; 0.8090169943749475 0.5877852522924731 -0.08; 0.30901699437494745 0.9510565162951535 -0.08; -0.30901699437494734 0.9510565162951536 -0.08; -0.8090169943749473 0.5877852522924732 -0.08; -1.0 1.2246467991473532e-16 -0.08; -0.8090169943749475 -0.587785252292473 -0.08; -0.30901699437494756 -0.9510565162951535 -0.08; 0.30901699437494723 -0.9510565162951536 -0.08; 0.8090169943749473 -0.5877852522924734 -

Excessive output truncated after 622671 bytes.

In [9]:
added_mass_r(r) = added_mass(radiation_solve(r, ω0)).Heave
added_mass_ω(ω) = added_mass(radiation_solve(r0, ω)).Heave

dA_dr  = DI.derivative(added_mass_r, fwd, r0)
# dA_dr′ = DI.derivative(added_mass_r, rev, r0)   # first Enzyme call compiles
# dA_dω  = DI.derivative(added_mass_ω, fwd, ω0)
# dA_dω′ = DI.derivative(added_mass_ω, rev, ω0)

# @printf("dA₃₃/dr   ForwardDiff %.6f   Enzyme %.6f\n", dA_dr, dA_dr′)
# @printf("dA₃₃/dω   ForwardDiff %.6f   Enzyme %.6f\n", dA_dω, dA_dω′)

NaN

In [5]:

# params_all = (wave_frequencies=[ω0], radiating_dofs=[:Heave], wave_directions=[β0])

# data = hydrodynamic_coefficients(body(r0), params_all, formulation)
# vec(data.added_mass)
# vec(data.radiation_damping)
# real(vec(data.excitation_force))

In [6]:
# A_of_r(r) = only(hydrodynamic_coefficients(body(r), params_all, formulation).added_mass)
# fex_of_r(r) = real(only(hydrodynamic_coefficients(body(r), params_all, formulation).excitation_force))

In [7]:
# A_of_r(r0)

In [8]:
# dA_dr = DI.derivative(A_of_r, fwd, r0)
# dfex_dr = DI.derivative(fex_of_r, fwd, r0)

# println("dA_dr: $dA_dr")
# println("dfex_dr: $dfex_dr")